In [2]:
FORCE_CPU = False
FRAME_STRIDE = 4
BATCH_SIZE = 1

In [3]:
import os
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
from ultralytics import YOLO
import cv2
import json

In [4]:
model = YOLO("modelos/YOLOv5-32.pt", verbose=False)

In [5]:
def process():
    results = model.predict("San FF02 Cerdocyon_thous (12).AVI", stream=True, verbose=False)
    b = []
    m = []
    p = []
    for r in results:
        boxes, masks, probs = r.boxes, r.masks, r.probs
        b.append(boxes)
        m.append(masks)
        p.append(probs)

In [6]:
%%timeit -n1 -r1
process()

15.1 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [7]:
def process2():
    results = model.predict("San FF02 Cerdocyon_thous (12).AVI", stream=True, verbose=False, batch=BATCH_SIZE, vid_stride=FRAME_STRIDE)
    b = []
    m = []
    p = []
    for r in results:
        boxes, masks, probs = r.boxes, r.masks, r.probs
        b.append(boxes)
        m.append(masks)
        p.append(probs)
    del results
    torch.cuda.empty_cache()
    return b, m, p

In [8]:
%%timeit -n1 -r1
b, m, p = process2()

3.67 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [9]:
b, m, p = process2()

In [10]:
len(b)

92

In [11]:
confidences = [it.conf.cpu().numpy().tolist() if len(it.conf) > 0 else [] for it in b ]

In [12]:
len(confidences)

92

In [13]:
bboxes = []
for it in b:
    if len(it.conf) > 0:
        xyxyn = it[0][0].xyxyn.cpu().numpy()
        for animal in xyxyn:
            #print(animal)
            animal = animal.tolist()
            bboxes.append([])
            bboxes[-1].append({
                "x0" : animal[0],
                "y0" : animal[1],
                "x1" : animal[2],
                "y1" : animal[3]
            })
    else:
        bboxes.append([{}])

In [14]:
bboxes

[[{'x0': 0.39285483956336975,
   'y0': 0.1668475717306137,
   'x1': 0.6664971709251404,
   'y1': 0.6348559856414795}],
 [{'x0': 0.41021451354026794,
   'y0': 0.11018755286931992,
   'x1': 0.6666542887687683,
   'y1': 0.6609383821487427}],
 [{'x0': 0.4235888421535492,
   'y0': 0.06538400053977966,
   'x1': 0.639224648475647,
   'y1': 0.7347856760025024}],
 [{'x0': 0.44162511825561523,
   'y0': 0.024431610479950905,
   'x1': 0.6611899733543396,
   'y1': 0.9267780184745789}],
 [{'x0': 0.44308310747146606,
   'y0': 0.0138846505433321,
   'x1': 0.7112938761711121,
   'y1': 0.9357449412345886}],
 [{'x0': 0.4482131898403168,
   'y0': 0.00150290597230196,
   'x1': 0.8016020059585571,
   'y1': 0.9591770172119141}],
 [{'x0': 0.4465516209602356,
   'y0': 0.006083509884774685,
   'x1': 0.9614101648330688,
   'y1': 0.9579278230667114}],
 [{'x0': 0.4858972728252411,
   'y0': 0.00366143137216568,
   'x1': 1.0,
   'y1': 0.9682736396789551}],
 [{'x0': 0.5560480356216431,
   'y0': 0.0017458598595112562,